In [13]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_core.runnables import RunnableSequence, RunnableLambda
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from langchain_core.prompts import PromptTemplate
from IPython.display import display, Markdown
from dotenv import load_dotenv
import os

load_dotenv()
GOOGLE_GEMINI_MODEL2 = os.getenv("GOOGLE_GEMINI_MODEL2")
GOOGLE_GEMINI_MODEL1 = os.getenv("GOOGLE_GEMINI_MODEL1")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY_NEW")

In [2]:
google_model_args = {
    "google_api_key" : GOOGLE_API_KEY,
    "model" : GOOGLE_GEMINI_MODEL2,
}
model = ChatGoogleGenerativeAI(**google_model_args)

In [6]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

pydantic_parser = PydanticOutputParser(
    name="movie",
    pydantic_object=Movie,
)
str_parser = StrOutputParser()
prompt = RunnableLambda(lambda movie_name : f"{movie_name} ")
format_instructions = pydantic_parser.get_format_instructions()
print(format_instructions)
llm_output = '{"title": "Alice", "year": 30, "director": "Alice", "rating": 5.2}'
output = pydantic_parser.parse(llm_output)
print({f"{output=}"})

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"description": "A movie with details.", "properties": {"title": {"description": "The title of the movie", "title": "Title", "type": "string"}, "year": {"description": "The year the movie was released", "title": "Year", "type": "integer"}, "director": {"description": "The director of the movie", "title": "Director", "type": "string"}, "rating": {"description": "The movie's rating out of 10", "title": "Rating", "type": "number"}}, "required": ["title", "year", "director", "rating"]}
```
{"output=Movie(title='Alice', year=30, director='Alice', r

In [11]:
prompt1 = PromptTemplate(
    template = "Write a paragraph on the movie {movie_name}",
    input_variables=["movie_name"],
    validate_template=True,
)
prompt2 = PromptTemplate(
    template = "Summerize the below paragraph in few lines.\n {para}",
    input_variables=["para"],
    validate_template=True,
)
runn = prompt1 | model | str_parser | prompt2 | model | str_parser
ans = runn.invoke("inception")

In [14]:
display(Markdown(ans))

Christopher Nolan's *Inception* features Leonardo DiCaprio as Dom Cobb, an extractor skilled in stealing secrets from dreams, who is tasked with the near-impossible opposite: planting an idea deep within a target's mind. This high-stakes mission unfolds across multi-layered dream levels, blending breathtaking action with profound psychological exploration that questions reality and memory. Anchored by Cobb's personal journey, the film's intricate plot, stunning visuals, and ambiguous ending solidify it as an unforgettable cinematic experience.